In [1]:
import numpy as np
from transformers import pipeline

# Thougts on improvement and expansion of scope
""" Implement Conditional statements to ensure all parties give arguments that follow the certain predefined conditions.(eg: statements must not contradict the geneva conventions) """
""" Implement grounding nodes as evidence for certain claims """
""" Implement verifiability for uncertain claims: Upload report or pdf that can be used for verifying claims (also can use claimify) """
""" Diplomatic use case: Ensure statement is consistent with old statements(done) and does not contradicts ally's stance """
""" Acceptability of a node (Dung Smantics). A node N is said to be acceptable wrt set S if all attackers of N have attackers in set S -> 
This can be used to identify the claims that are unsupported and to be focused on during effective argumentation"""


def find_cycles_adj_matrix(adj_matrix):
    n = len(adj_matrix)
    visited = [False] * n
    stack = [False] * n
    cycles = []
    
    def dfs(v, path):
        visited[v] = True
        stack[v] = True
        path.append(v)

        for u in range(n):
            if adj_matrix[v][u]:  # edge exists
                if not visited[u]:
                    dfs(u, path)
                elif stack[u]:
                    cycle_start = path.index(u)
                    cycles.append(path[cycle_start:].copy())

        stack[v] = False
        path.pop()

    for node in range(n):
        if not visited[node]:
            dfs(node, [])

    return cycles



In [2]:
"""Give real world usage examples"""

uniform_debate_template = [
    "Uniforms promote equality",
    "Uniforms restrict personal expression",
    "Uniforms reduce bullying",
    "Uniforms do not promote equality",
    "Uniforms are not non beneficial"
]

uniform_initial_template = [
    "Uniforms are benefecial",
    "Uiforms are not benefecial"
]

animal_debate_template = [
    "killing animals is bad",
    "animals are tasty to eat"
]
animal_initial_template = [
    "killing living things is ok",
    "killing living things is not ok"
]

contradiction_check_template = ["i am awesome", "noone cares about me", "i suck"]

# Complex Circular Reasoning (Cycle)
circular_reasoning_nodes = [
    "This law is fair because it promotes justice.",
    "It promotes justice because the law ensures fairness.",
    "Ensuring fairness is the law's purpose because lawmakers intended it.",
    "Lawmakers intended the law to promote justice, which is fair."  # loops back to first node
]

# Complex Contradiction (Multi-branch)
contradiction_nodes = [
    "All industrial pollution should be banned to protect nature.",
    "Factory A’s pollution are desirable because it creates jobs.",  # contradicts first node
    "Factory B’s pollution are harmful and should be stopped.",       # consistent with first node
    "We should prioritize human welfare over environmental rules."    # introduces tension
]

# Combined Cycle + Contradiction
combined_cycle_contradiction_nodes = [
    "Hunting is unethical because killing is wrong.",
    "Killing for population control is ethical.",
    "Population control is necessary, so hunting is acceptable.",
    "Hunting is unethical because killing is wrong."  # cycle with contradiction
]


EMPTY_TEMPLATE = []

In [3]:
LABEL_DICT = {'ENTAILMENT': 1,'CONTRADICTION': -1, "NEUTRAL": 0}
NUM_SPEAKER = 2
DEBATE_TEMPLATE = circular_reasoning_nodes
INITIAL_TEMPLATE = EMPTY_TEMPLATE
STANCE_MODEL =  pipeline("text-classification", model="roberta-large-mnli")

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [4]:
class Fallacy_Checker():
    
    def __init__(self, A_t, C_t):
        self.A_t = A_t
        self.C_t = C_t
        self.speaker_A_t = None
        if(self.C_t != []):
            self.C_t =  np.array(self.C_t) #ARGS x NUM_SPEAKER
            speaker_arg_mask = np.transpose(self.C_t , (1,0)) #NUM_SPEAKER x ARGS
            self.A_t = np.array(self.A_t) #ARGS X ARGS
            self.speaker_A_t =  np.einsum('ij,jk->ijk', speaker_arg_mask, self.A_t) #NUM_SPEAKER x ARGS
        
    def contradiction(self):
        contradictions = {}
        for speaker in range(NUM_SPEAKER):
            for i in range(len(self.speaker_A_t[speaker])):
                for j in range(i, len(self.speaker_A_t[speaker])):
                    if((self.speaker_A_t[speaker][i][j], self.speaker_A_t[speaker][j][i]) in ((1,-1), (-1,-1), (-1,1))):
                        if(speaker in contradictions):
                            contradictions[speaker].append([i,j])
                        else:
                            contradictions[speaker] = [[i,j]]
                    
        if(len(contradictions)>0):
            return contradictions
        return None
    
    def circular_reasoning(self):
        cycles = {}
        
        for speaker in range(NUM_SPEAKER):
            x = self.speaker_A_t[speaker]
            x = np.where(x == 1, x, 0)
            cycles[speaker] = find_cycles_adj_matrix(x)
            
        if(len(cycles)>0):
            return cycles
        return None
        

class Debate():
    def __init__(self, stance_model = STANCE_MODEL):
        self.claims = []
        self.num_speaker = 0
        self.n = len(self.claims)
        
        self.A_t = [[0 for _ in range(self.n)] for _ in range(self.n)]
        self.C_t = [[0 for _ in range(self.num_speaker)] for _ in range(self.n)]

    
        
        self.nli_model = stance_model
        
        self.fallacy_checker = Fallacy_Checker(self.A_t, self.C_t)
        self.circular_reasoning = None
        self.contradictions = None

        
    def add_claim(self, speaker:int, claim:str):
        #Check for new speaker
        if(speaker>=self.num_speaker):
            assert speaker == self.num_speaker
            for ctk in self.C_t:
                ctk.append(0)
            self.num_speaker+=1
        
        self.A_t.append([0 for _ in range(self.n)])
        self.n += 1
        self.claims.append(claim)
        self.C_t.append([0 for _ in range(self.num_speaker)])
        self.update_A_t()
        # print(len(self.C_t))
        self.C_t[-1][speaker] = 1
    
    def update_A_t(self):
        claim_combinations = []
        for i in self.claims:
            for j in self.claims:
                claim_combinations.append(f"{i} [SEP] {j}.")  
        
        pred = self.nli_model(claim_combinations)
        for idx, p in enumerate(pred):
            
            row = idx // self.n
            col = idx % self.n
            if col == self.n-1:
                self.A_t[row].append(LABEL_DICT[p["label"]])
            else:
                self.A_t[row][col] = LABEL_DICT[p["label"]]
    
    def check_fallacy(self):
        self.fallacy_checker = Fallacy_Checker(self.A_t, self.C_t)
        self.circular_reasoning = self.fallacy_checker.circular_reasoning()
        self.contradictions = self.fallacy_checker.contradiction()
        
        if(self.circular_reasoning):
            print("|----------The arguments have circular reasoning ----------|")
            print(self.circular_reasoning)
        
        if(self.contradictions):
            print("|----------The arguments have contradictions----------|")
            print(self.contradictions)
    
    def show_fallacy(self):
        if(self.circular_reasoning):
            for speaker in self.circular_reasoning:
                print(f"|---------Circular Reasoning Fallacy of speaker: {speaker}--------|")
                for fallacies in self.circular_reasoning[speaker]:
                    s = ""
                    if(len(fallacies)>1):
                        s = " <-- "
                        for claim_idx in fallacies:
                            s += self.claims[claim_idx]
                            s += " --> "
                    if(s != ""):
                        print(s)
        
        if(self.contradictions):
            for speaker in self.contradictions:
                print(f"|---------Contradiction Fallacy of speaker: {speaker}--------------|")
                for fallacies in self.contradictions[speaker]:
                    print(f" {self.claims[fallacies[0]]} != {self.claims[fallacies[1]]} ")
                    
                    


In [5]:
circular_reasoning_nodes = [
    [0, "This law is fair because it promotes justice."],
    [0,"It promotes justice because the law ensures fairness."],
    [0,"Ensuring fairness is the law's purpose because lawmakers intended it."],
    [0,"Lawmakers intended the law to promote justice, which is fair."]  # loops back to first node
]

contradiction_nodes = [
    [0,"All industrial pollution should be banned to protect nature."],
    [1,"All industrial pollution should be banned to protect nature."],
    [1,"Factory A’s pollution are desirable because it creates jobs."],  # contradicts first node
    [0,"Factory B’s pollution are harmful and should be stopped."],       # consistent with first node
    [0,"We should prioritize human welfare over environmental rules."]    # introduces tension
]

a = [
    [0,"I am happy"],
    [0,"I am sad"]
]
a = [
    [0, "Climate Change is primarily through activities like burning fossil fuels"],
    [0, "Some scientists argue that climate change is a part of the natural cycle of the Earth"],
    [0, "If climate Change was natural, CO2 levels would not have risen so sharply after industrialization"],
    [0, "Therefore, human activities are the main driver of recent climate change"],
    [0, "But if human activities are the main driver, then stopping emmissino should immediately stop all climate change"],
    [0, "So human activities cannot be the main driver of climate change"]
]
NUM_SPEAKER = 1
ROUNDS = len(DEBATE_TEMPLATE)
debate = Debate()

In [6]:
speaker = 0
template = a
for round in range(len(template)):
    # claim = str(input(f"Enter claim for speaker {speaker}: "))
    speaker, claim = template[round]
    debate.add_claim(speaker, claim)

In [7]:
#print adjacency matrix that shows relationship between all claims
for i in debate.A_t:
    print(i)
print()
#Check claims and relationship between claims 
for i,j in zip(debate.A_t, debate.claims):
    print(j,i)

[1, 0, 0, 1, 0, 0]
[0, 1, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0]
[0, 0, 0, 1, 0, -1]
[0, 0, 0, 0, 1, 0]
[0, 0, 0, -1, 0, 1]

Climate Change is primarily through activities like burning fossil fuels [1, 0, 0, 1, 0, 0]
Some scientists argue that climate change is a part of the natural cycle of the Earth [0, 1, 0, 0, 0, 0]
If climate Change was natural, CO2 levels would not have risen so sharply after industrialization [0, 0, 1, 0, 0, 0]
Therefore, human activities are the main driver of recent climate change [0, 0, 0, 1, 0, -1]
But if human activities are the main driver, then stopping emmissino should immediately stop all climate change [0, 0, 0, 0, 1, 0]
So human activities cannot be the main driver of climate change [0, 0, 0, -1, 0, 1]


In [8]:
debate.check_fallacy()

|----------The arguments have circular reasoning ----------|
{0: [[0], [3], [1], [2], [4], [5]]}
|----------The arguments have contradictions----------|
{0: [[3, 5]]}


In [9]:
debate.show_fallacy()

|---------Circular Reasoning Fallacy of speaker: 0--------|
|---------Contradiction Fallacy of speaker: 0--------------|
 Therefore, human activities are the main driver of recent climate change != So human activities cannot be the main driver of climate change 


In [10]:

C_t =  np.array(debate.C_t) #ARGS x NUM_SPEAKER
speaker_arg_mask = np.transpose(C_t , (1,0)) #NUM_SPEAKER x ARGS
A_t = np.array(debate.A_t) #ARGS X ARGS
speaker_A_t =  np.einsum('ij,jk->ijk', speaker_arg_mask,A_t) #NUM_SPEAKER x ARGS
print(speaker_A_t)
print(speaker_arg_mask)

[[[ 1  0  0  1  0  0]
  [ 0  1  0  0  0  0]
  [ 0  0  1  0  0  0]
  [ 0  0  0  1  0 -1]
  [ 0  0  0  0  1  0]
  [ 0  0  0 -1  0  1]]]
[[1 1 1 1 1 1]]
